In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import os
import glob
import gc

# each file is ~1gb

folder = '/glade/derecho/scratch/sarahryu/leap2026/nodeep_1deg/'
files = glob.glob(os.path.join(folder, "*.nc"))

print(len(files))

562


In [2]:
def clean_ds(ds):

    tin = ds['T']
    tin.name = 'Tin'
    
    qin = ds['Q'] + ds['CLDLIQ'] + ds['CLDICE'] ### remember to add attributes later!!!
    
    qin.name = "qin"
    qin.attrs = {**ds["Q"].attrs,
                       "long_name": "vapor + cloud liquid + cloud ice",
                       "units": "kg/kg"}
    
    uin = ds['U']
    uin.name = 'Uin'
    
    vinMinusSH = ds["V"] * xr.where(ds["lat"] < 0, -1, 1)
    
    vinMinusSH.name = "vinMinusSH"
    vinMinusSH.attrs = {**ds["V"].attrs,
                       "long_name": "V (vertical wind speed) with sign flipped in southern hemisphere",
                       "units": "m/s"}
    
    
    usurf = ds['U10']
    usurf.name = 'usurf'


    # lat + surface, topography vars

    landfrac = ds['LANDFRAC']

    icefrac = ds['ICEFRAC']

    phis = ds['PHIS']

    solin = ds['SOLIN']

    

    dtcond = ds['DTCOND']
    dcq = ds['DCQ']

    return xr.merge([
        tin,
        qin,
        uin,
        vinMinusSH,
        usurf,
        landfrac,
        icefrac,
        phis,
        solin,
        dtcond,
        dcq
    ])


In [3]:

from tqdm import tqdm

target_folder = '/glade/derecho/scratch/sarahryu/leap2026/nodeep_1deg_clean/'


for file in tqdm(files):

    target_file = os.path.join(target_folder, os.path.basename(file))
    if os.path.exists(target_file):
        continue

    ds_clean = clean_ds(xr.open_dataset(file))
    ds_clean.to_netcdf(target_file)
    del ds_clean
    gc.collect()


### each new file is ~200mb

100%|██████████| 562/562 [01:14<00:00,  7.56it/s]
